In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import numpy as np
import random
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score, matthews_corrcoef, balanced_accuracy_score
import pandas as pd
import seaborn as sns
import time

# Set random seeds for reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# Define the device for computation
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Mount Google Drive to save and load the model (if using Google Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    base_path = '/content/drive/My Drive/FYP'
except ImportError:
    base_path = '.'  # Use current directory if not in Colab

# Define the folder to save model checkpoints
date_str = datetime.now().strftime('%Y%m%d')
checkpoint_folder = f'{base_path}/VGGModel/HQ3latest_{date_str}/'
os.makedirs(checkpoint_folder, exist_ok=True)

# Data Augmentation for Training Set
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),  # VGG expects 224x224 input size
    transforms.RandomHorizontalFlip(p=0.5),  # Flip left/right
    transforms.RandomVerticalFlip(p=0.3),  # Flip up/down
    transforms.RandomRotation(degrees=10),  # Small rotation
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),  # Slight shifts
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),  # Small color changes
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Simple resizing for validation and test sets
transform_val_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Load datasets
data_path = f'{base_path}/Dataset/HQ3/train'
test_data_path = f'{base_path}/Dataset/HQ3/test'
train_dataset = datasets.ImageFolder(data_path, transform=transform_train)
test_dataset = datasets.ImageFolder(test_data_path, transform=transform_val_test)

# Split the dataset into 70% training and 30% validation
train_size = int(0.7 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

# Set validation set transformation explicitly
val_dataset.dataset.transform = transform_val_test

# Create data loaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

# Use Pre-trained VGG-16 model and modify it for binary classification
class VGG16Modified(nn.Module):
    def __init__(self, num_classes=2, dropout_rate=0.4):
        super(VGG16Modified, self).__init__()
        from torchvision.models import vgg16, VGG16_Weights

        # Load pre-trained VGG16 model
        self.vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)

        # Replace the classifier with a custom classification layer
        num_ftrs = self.vgg.classifier[6].in_features
        self.vgg.classifier[6] = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_ftrs, num_classes)
        )

    def forward(self, x):
        return self.vgg(x)

# Initialize the VGG model
model = VGG16Modified().to(device)

# Define loss function and optimizer with L2 regularization (weight decay)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=3, verbose=True)

# Early stopping class
class EarlyStopping:
    """Early stops the training if validation loss OR accuracy doesn't improve after a given patience."""
    def __init__(self, patience=5, verbose=True, delta=0, path='checkpoint.pth', trace_func=print, monitor='loss'):
        """
        Args:
            patience (int): How long to wait after last improvement.
            verbose (bool): If True, prints a message for each improvement.
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
            path (str): Path for the checkpoint to be saved to.
            trace_func (function): Function to trace progress.
            monitor (str): Metric to monitor for early stopping ('loss' or 'accuracy').
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.val_acc_max = 0
        self.delta = delta
        self.path = path
        self.trace_func = trace_func
        self.monitor = monitor  # 'loss' or 'accuracy'
        self.improvement = False
        self.stopped_epoch = 0

    def __call__(self, val_loss, val_acc, model, epoch):
        if self.monitor == 'loss':
            score = -val_loss
        else:  # monitor accuracy
            score = val_acc

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, val_acc, model)
            self.improvement = True
        elif score < self.best_score + self.delta:
            self.counter += 1
            self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            self.improvement = False
            if self.counter >= self.patience:
                self.early_stop = True
                self.stopped_epoch = epoch
                self.trace_func(f'Early stopping triggered at epoch {epoch}')
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, val_acc, model)
            self.counter = 0
            self.improvement = True

    def save_checkpoint(self, val_loss, val_acc, model):
        '''Saves model when validation loss decreases or accuracy increases.'''
        if self.verbose:
            if self.monitor == 'loss':
                self.trace_func(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
                self.val_loss_min = val_loss
            else:
                self.trace_func(f'Validation accuracy increased ({self.val_acc_max:.2f}% --> {val_acc:.2f}%).  Saving model ...')
                self.val_acc_max = val_acc
        torch.save(model.state_dict(), self.path)

def calculate_metrics(y_true, y_pred, y_scores):
    """Calculate various classification metrics."""
    metrics = {}
    metrics['accuracy'] = accuracy_score(y_true, y_pred) * 100
    metrics['balanced_accuracy'] = balanced_accuracy_score(y_true, y_pred)
    metrics['precision'] = precision_score(y_true, y_pred, average='binary')
    metrics['recall'] = recall_score(y_true, y_pred, average='binary')
    metrics['f1'] = f1_score(y_true, y_pred, average='binary')
    metrics['mcc'] = matthews_corrcoef(y_true, y_pred)

    # For ROC-AUC and PR-AUC, we need probabilities
    # Ensure y_scores are probabilities of the positive class
    metrics['roc_auc'] = roc_auc_score(y_true, y_scores)
    metrics['pr_auc'] = average_precision_score(y_true, y_scores)

    return metrics

def evaluate(model, loader, criterion):
    """Evaluate model and return predictions, true labels, losses and metrics."""
    model.eval()
    all_preds = []
    all_labels = []
    all_scores = []
    total_loss = 0.0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Get predicted class and probability scores
            probs = torch.softmax(outputs, dim=1)
            pos_probs = probs[:, 1].cpu().numpy()  # Probability of positive class
            _, predicted = torch.max(outputs.data, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_scores.extend(pos_probs)

    # Calculate metrics
    metrics = calculate_metrics(all_labels, all_preds, all_scores)
    metrics['loss'] = total_loss / len(loader)

    return metrics, all_preds, all_labels, all_scores

def train(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=30):
    """Train the model and track metrics with both accuracy and loss-based early stopping."""
    # Initialize lists to store metrics
    metrics_df = pd.DataFrame(columns=['epoch', 'training_loss', 'val_loss', 'val_accuracy',
                                       'val_balanced_accuracy', 'val_precision', 'val_recall',
                                       'val_f1', 'val_mcc', 'val_roc_auc', 'val_pr_auc',
                                       'lr', 'time_taken'])

    # Paths for different model checkpoints
    checkpoint_model_path = os.path.join(checkpoint_folder, f"checkpoint_model_vgg_{date_str}.pth")
    best_acc_model_path = os.path.join(checkpoint_folder, f"best_acc_model_vgg_{date_str}.pth")
    best_loss_model_path = os.path.join(checkpoint_folder, f"best_loss_model_vgg_{date_str}.pth")

    # Initialize early stopping for both accuracy and loss monitoring
    early_stopping_acc = EarlyStopping(
        patience=5,
        verbose=True,
        path=best_acc_model_path,
        monitor='accuracy'
    )

    early_stopping_loss = EarlyStopping(
        patience=3,  # Shorter patience for loss monitoring
        verbose=True,
        path=best_loss_model_path,
        monitor='loss'
    )

    # Tracking variables
    start_time = time.time()
    epoch_times = []

    for epoch in range(epochs):
        epoch_start = time.time()

        # Check if early stopping triggered by either monitor
        if early_stopping_acc.early_stop or early_stopping_loss.early_stop:
            stopped_by = "accuracy" if early_stopping_acc.early_stop else "loss"
            print(f"Early stopping triggered based on validation {stopped_by}")
            break

        # Training phase
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            optimizer.step()

            running_loss += loss.item()

        # Calculate training loss
        train_loss = running_loss / len(train_loader)

        # Validation phase
        val_metrics, val_preds, val_labels, val_scores = evaluate(model, val_loader, criterion)
        val_loss = val_metrics['loss']
        val_acc = val_metrics['accuracy']

        # Update learning rate scheduler based on validation accuracy
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_acc)

        # Apply early stopping checks
        early_stopping_acc(val_loss, val_acc, model, epoch)
        early_stopping_loss(val_loss, val_acc, model, epoch)

        # Save regular checkpoint every epoch
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_acc': val_acc
        }, checkpoint_model_path)

        # Calculate time taken for epoch
        epoch_end = time.time()
        epoch_time = epoch_end - epoch_start
        epoch_times.append(epoch_time)

        # Print epoch results with detailed early stopping information
        improvement_acc = "✓" if early_stopping_acc.improvement else "✗"
        improvement_loss = "✓" if early_stopping_loss.improvement else "✗"

        print(f"Epoch [{epoch+1}/{epochs}] - Time: {epoch_time:.2f}s - "
              f"Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f} [{improvement_loss}], "
              f"Val Acc: {val_acc:.2f}% [{improvement_acc}], "
              f"Val F1: {val_metrics['f1']:.4f}, LR: {current_lr:.8f}")

        # Add metrics to dataframe
        metrics_df = pd.concat([metrics_df, pd.DataFrame({
            'epoch': [epoch + 1],
            'training_loss': [train_loss],
            'val_loss': [val_loss],
            'val_accuracy': [val_metrics['accuracy']],
            'val_balanced_accuracy': [val_metrics['balanced_accuracy']],
            'val_precision': [val_metrics['precision']],
            'val_recall': [val_metrics['recall']],
            'val_f1': [val_metrics['f1']],
            'val_mcc': [val_metrics['mcc']],
            'val_roc_auc': [val_metrics['roc_auc']],
            'val_pr_auc': [val_metrics['pr_auc']],
            'lr': [current_lr],
            'time_taken': [epoch_time]
        })], ignore_index=True)

    # Training complete
    total_time = time.time() - start_time
    avg_epoch_time = np.mean(epoch_times)

    # Save early stopping information
    early_stopping_info = pd.DataFrame({
        'stopped_epoch_acc': [early_stopping_acc.stopped_epoch if early_stopping_acc.early_stop else -1],
        'stopped_epoch_loss': [early_stopping_loss.stopped_epoch if early_stopping_loss.early_stop else -1],
        'best_val_acc': [early_stopping_acc.val_acc_max],
        'best_val_loss': [early_stopping_loss.val_loss_min],
        'patience_acc': [early_stopping_acc.patience],
        'patience_loss': [early_stopping_loss.patience],
        'total_epochs': [len(metrics_df)],
        'total_time_sec': [total_time],
        'avg_epoch_time_sec': [avg_epoch_time]
    })
    early_stopping_info.to_csv(os.path.join(checkpoint_folder, f"early_stopping_info_{date_str}.csv"), index=False)

    # Save metrics to CSV
    metrics_csv_path = os.path.join(checkpoint_folder, f"metrics_{date_str}.csv")
    metrics_df.to_csv(metrics_csv_path, index=False)
    print(f"Training metrics saved to {metrics_csv_path}")

    # Plot and save training curves
    plot_training_curves(metrics_df, checkpoint_folder)

    # Determine which early stopping method was triggered first
    stopped_by = None
    if early_stopping_acc.early_stop and early_stopping_loss.early_stop:
        if early_stopping_acc.stopped_epoch <= early_stopping_loss.stopped_epoch:
            stopped_by = "accuracy"
        else:
            stopped_by = "loss"
    elif early_stopping_acc.early_stop:
        stopped_by = "accuracy"
    elif early_stopping_loss.early_stop:
        stopped_by = "loss"
    else:
        stopped_by = None  # No early stopping

    print(f"\nTraining completed in {total_time:.2f} seconds ({len(metrics_df)} epochs, avg {avg_epoch_time:.2f}s per epoch)")
    if stopped_by:
        print(f"Early stopping triggered by {stopped_by} monitoring")

    return metrics_df, best_acc_model_path, best_loss_model_path

def plot_training_curves(metrics_df, save_folder):
    """Plot and save training curves with early stopping indicators."""
    # Create figure with 2x2 subplots
    plt.figure(figsize=(20, 15))

    # Plot 1: Training vs Validation Loss
    plt.subplot(2, 2, 1)
    plt.plot(metrics_df['epoch'], metrics_df['training_loss'], 'b-', label='Training Loss')
    plt.plot(metrics_df['epoch'], metrics_df['val_loss'], 'r-', label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Training vs Validation Loss')
    plt.legend()
    plt.grid(True)

    # Plot 2: Validation Accuracy
    plt.subplot(2, 2, 2)
    plt.plot(metrics_df['epoch'], metrics_df['val_accuracy'], 'g-')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.title('Validation Accuracy')
    plt.grid(True)

    # Plot 3: Validation F1, Precision, Recall
    plt.subplot(2, 2, 3)
    plt.plot(metrics_df['epoch'], metrics_df['val_f1'], 'c-', label='F1 Score')
    plt.plot(metrics_df['epoch'], metrics_df['val_precision'], 'm-', label='Precision')
    plt.plot(metrics_df['epoch'], metrics_df['val_recall'], 'y-', label='Recall')
    plt.xlabel('Epochs')
    plt.ylabel('Score')
    plt.title('Validation F1, Precision, Recall')
    plt.legend()
    plt.grid(True)

    # Plot 4: Additional metrics
    plt.subplot(2, 2, 4)
    plt.plot(metrics_df['epoch'], metrics_df['val_roc_auc'], 'b-', label='ROC-AUC')
    plt.plot(metrics_df['epoch'], metrics_df['val_pr_auc'], 'r-', label='PR-AUC')
    plt.plot(metrics_df['epoch'], metrics_df['val_mcc'], 'g-', label='MCC')
    plt.xlabel('Epochs')
    plt.ylabel('Score')
    plt.title('Validation ROC-AUC, PR-AUC, and MCC')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(save_folder, f"training_curves_{date_str}.png"), dpi=300)

    # Create additional plots specifically for early stopping visualization
    plt.figure(figsize=(15, 10))

    # Plot 1: Learning Rate over epochs
    plt.subplot(2, 1, 1)
    plt.plot(metrics_df['epoch'], metrics_df['lr'], 'b-')
    plt.yscale('log')
    plt.xlabel('Epochs')
    plt.ylabel('Learning Rate (log scale)')
    plt.title('Learning Rate Schedule')
    plt.grid(True)

    # Plot 2: Early stopping focus - Training vs Validation Loss
    plt.subplot(2, 1, 2)
    plt.plot(metrics_df['epoch'], metrics_df['training_loss'], 'b-', label='Training Loss')
    plt.plot(metrics_df['epoch'], metrics_df['val_loss'], 'r-', label='Validation Loss')

    # Add min validation loss point
    min_val_loss_idx = metrics_df['val_loss'].idxmin()
    min_val_loss = metrics_df.loc[min_val_loss_idx, 'val_loss']
    min_val_loss_epoch = metrics_df.loc[min_val_loss_idx, 'epoch']
    plt.scatter(min_val_loss_epoch, min_val_loss, c='red', s=100, marker='o', label=f'Min Val Loss: {min_val_loss:.4f}')

    # Add best validation accuracy point
    max_val_acc_idx = metrics_df['val_accuracy'].idxmax()
    max_val_acc = metrics_df.loc[max_val_acc_idx, 'val_accuracy']
    max_val_acc_epoch = metrics_df.loc[max_val_acc_idx, 'epoch']
    plt.scatter(max_val_acc_epoch, metrics_df.loc[max_val_acc_idx, 'val_loss'],
               c='green', s=100, marker='s', label=f'Best Val Acc: {max_val_acc:.2f}%')

    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_folder, f"early_stopping_focus_{date_str}.png"), dpi=300)
    plt.close()

    # Plot confusion matrices for best models
    print("\nEvaluating best accuracy model on validation set:")
    model.load_state_dict(torch.load(best_acc_model_path))
    val_metrics_acc, val_preds_acc, val_labels_acc, _ = evaluate(model, val_loader, criterion)

    print("\nEvaluating best loss model on validation set:")
    model.load_state_dict(torch.load(best_loss_model_path))
    val_metrics_loss, val_preds_loss, val_labels_loss, _ = evaluate(model, val_loader, criterion)

    # Compare the models
    print(f"\nComparison - Best Accuracy Model vs Best Loss Model:")
    print(f"Best Accuracy Model - Val Acc: {val_metrics_acc['accuracy']:.2f}%, Val Loss: {val_metrics_acc['loss']:.6f}")
    print(f"Best Loss Model - Val Acc: {val_metrics_loss['accuracy']:.2f}%, Val Loss: {val_metrics_loss['loss']:.6f}")

    return metrics_df, best_acc_model_path, best_loss_model_path

def plot_confusion_matrix(y_true, y_pred, class_names, save_folder, dataset_name):
    """Plot and save confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'Confusion Matrix - {dataset_name.capitalize()} Set')
    plt.tight_layout()
    plt.savefig(os.path.join(save_folder, f"confusion_matrix_{dataset_name}_{date_str}.png"), dpi=300)
    plt.close()

def test_model(model, test_loader, criterion, model_name="best"):
    """Evaluate the model on the test set and visualize results."""
    print(f"\nEvaluating {model_name} model on test set...")
    test_metrics, test_preds, test_labels, test_scores = evaluate(model, test_loader, criterion)

    # Print test metrics
    print(f"\nTest Set Metrics ({model_name} model):")
    print(f"Loss: {test_metrics['loss']:.6f}")
    print(f"Accuracy: {test_metrics['accuracy']:.2f}%")
    print(f"Balanced Accuracy: {test_metrics['balanced_accuracy']:.4f}")
    print(f"Precision: {test_metrics['precision']:.4f}")
    print(f"Recall: {test_metrics['recall']:.4f}")
    print(f"F1 Score: {test_metrics['f1']:.4f}")
    print(f"MCC: {test_metrics['mcc']:.4f}")
    print(f"ROC-AUC: {test_metrics['roc_auc']:.4f}")
    print(f"PR-AUC: {test_metrics['pr_auc']:.4f}")

    # Plot confusion matrix for test set
    plot_confusion_matrix(test_labels, test_preds, ['Normal', 'Tumor'], checkpoint_folder, f"test_{model_name}")

    # Save test metrics to CSV
    test_metrics_df = pd.DataFrame({metric: [value] for metric, value in test_metrics.items()})
    test_metrics_df.to_csv(os.path.join(checkpoint_folder, f"test_metrics_{model_name}_{date_str}.csv"), index=False)

    return test_metrics

# ------------------- Main Training & Testing Routine -------------------
if __name__ == "__main__":
    # Train the model and collect metrics
    metrics_df, best_acc_model_path, best_loss_model_path = train(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=30)

    # Load best accuracy model for testing
    model.load_state_dict(torch.load(best_acc_model_path))
    test_metrics_acc = test_model(model, test_loader, criterion, "best_accuracy")

    # Load best loss model for testing
    model.load_state_dict(torch.load(best_loss_model_path))
    test_metrics_loss = test_model(model, test_loader, criterion, "best_loss")

    # Compare test results between models
    print("\nTest Set Comparison - Best Accuracy Model vs Best Loss Model:")
    print(f"Best Accuracy Model - Test Acc: {test_metrics_acc['accuracy']:.2f}%, Test Loss: {test_metrics_acc['loss']:.6f}")
    print(f"Best Loss Model - Test Acc: {test_metrics_loss['accuracy']:.2f}%, Test Loss: {test_metrics_loss['loss']:.6f}")

    print(f"\nTraining and evaluation complete. Results saved to {checkpoint_folder}")

Using device: cpu
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
